# Phase 18+19: Feature Selection & Baseline Model Benchmark
## Univariate Filtering, Multicollinearity (VIF), Tree Importance & Honest Baseline Floor

**Quant Trading Bot — Phase 18+19 of 50 (Finishing Feature Engineering & Starting Core ML)**

### Executive Summary:
This milestone spans two interconnected deliverables:
1. **PART A (Phase 18) — Feature Selection & Importance Analysis**:
   - Defining rigorous prediction targets ($h=1$ day directional classification and forward returns) with strict no-lookahead assertions.
   - Univariate filtering via Pearson/Spearman rank correlation and Scikit-Learn Mutual Information.
   - Multicollinearity diagnostics using Variance Inflation Factor (VIF) and pairwise correlation clustering.
   - Constrained baseline Decision Tree feature importance ranking.
   - Generating a non-collinear recommended feature shortlist.
   - **Crucial Anti-Leakage Discipline**: Feature selection is executed **strictly on training-period data** (`2018`–`2023`), preventing test-period leakage.

2. **PART B (Phase 19) — Baseline Model Benchmarking**:
   - Establishing the performance floor that complex architectures (XGBoost in Phase 21, LSTMs in Phase 26) must beat.
   - Benchmarking against a **Naive Persistence Baseline** (predicting yesterday's direction repeats: $\hat{Y}_t = Y_{t-1}$).
   - Testing simple **Logistic Regression** and an **Interpretable Decision Tree** on both full and shortlisted features.
   - Honest, unvarnished quantitative read on directional predictability in daily equity markets.


In [2]:
import sys
import types
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if "matplotlib._c_internal_utils" not in sys.modules:
    try:
        import matplotlib._c_internal_utils
    except ImportError:
        sys.modules["matplotlib._c_internal_utils"] = types.ModuleType("matplotlib._c_internal_utils")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_pipeline.data_access import get_data_access
from src.features.feature_scaling import FeaturePipeline
from src.features.feature_selection import make_target, feature_selection_report
from src.models.baseline_model import (
    temporal_train_test_split,
    run_baseline_comparison,
    BaselineClassifier,
    NaivePersistenceModel,
    evaluate_classification
)

dal = get_data_access()
tickers = ["SPY", "AAPL", "MSFT"]
dfs = {}
for t in tickers:
    df = dal.get_ohlcv(t)
    if "date" in df.columns and not isinstance(df.index, pd.DatetimeIndex):
        df = df.set_index(pd.to_datetime(df["date"])).sort_index()
    dfs[t] = df
    print(f"{t:<5}: {len(df)} bars ({df.index[0].date()} to {df.index[-1].date()}) | Closes: ${df['close'].iloc[0]:.2f} -> ${df['close'].iloc[-1]:.2f}")



SPY  : 2177 bars (2018-01-02 to 2026-08-31) | Closes: $235.95 -> $767.05
AAPL : 2177 bars (2018-01-02 to 2026-08-31) | Closes: $40.23 -> $316.85
MSFT : 2177 bars (2018-01-02 to 2026-08-31) | Closes: $78.55 -> $507.29


## 1. Prediction Target Formulation (No-Lookahead Guarantee)

### Target Definition:
- **Task**: Next-day market direction classification ($h = 1$ trading day).
- **Formula**:
  $$R_{t, t+1} = \frac{P_{t+1} - P_t}{P_t}$$
  $$Y_t = \begin{cases} 1.0 & \text{if } R_{t, t+1} > 0 \\ 0.0 & \text{if } R_{t, t+1} \le 0 \end{cases}$$
- **Anti-Leakage Safeguard**:
  The target at time $t$ uses strictly future price $P_{t+1}$ via negative shift (`shift(-1)`). The final bar is guaranteed to be `NaN` because the next day's outcome is unobserved at historical dataset boundaries.


In [4]:
pipeline_features = [
    'mom_5d', 'mom_10d', 'mom_20d', 'mom_60d', 'mom_252d', 'roc_10', 'roc_20', 'rsi_14', 'macd_12_26_9', 'sma_50_200_spread',
    'zscore_10d', 'zscore_20d', 'zscore_50d', 'bb_pct_b_20_2', 'bb_bandwidth_20_2', 'ma_dist_atr_20', 'stoch_slow_k', 'half_life_120d',
    'garch_vol', 'garch_vol_annualized',
    'obv', 'vwap_20', 'adl', 'cmf_20', 'volume_roc_10', 'volume_zscore_20', 'amihud_illiquidity_20',
    'corwin_schultz_spread_20', 'roll_spread_20', 'vpin_proxy_20', 'garman_klass_vol_20', 'parkinson_vol_20',
    'regime_label', 'regime_prob_0', 'regime_prob_1', 'regime_entropy',
]

data_dict = {}
split_date = pd.Timestamp("2024-01-01")

for t in tickers:
    raw_df = dfs[t]
    # Build target series
    target_s = make_target(raw_df, horizon=1, task_type="classification")
    
    # Extract continuous features
    pipeline = FeaturePipeline(feature_names=pipeline_features, scaler_method="robust", max_ffill=5, drop_warmup=True)
    raw_feats = pipeline.extract_features(raw_df)
    clean_feats = pipeline.clean_features(raw_feats)
    
    # Align features and target
    common_idx = clean_feats.index.intersection(target_s.dropna().index)
    X_clean = clean_feats.loc[common_idx]
    y_clean = target_s.loc[common_idx]
    
    # Temporal chronological split: Train (2018-2023) and Test (2024-2026)
    train_mask = X_clean.index < split_date
    test_mask = X_clean.index >= split_date
    
    # Fit scaler strictly on train
    pipeline.scaler.fit(X_clean.loc[train_mask])
    X_train = pipeline.scaler.transform(X_clean.loc[train_mask])
    X_test = pipeline.scaler.transform(X_clean.loc[test_mask])
    
    y_train = y_clean.loc[train_mask]
    y_test = y_clean.loc[test_mask]
    
    data_dict[t] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
    }
    
    print(f"=== {t} Matrix Assembly ===")
    print(f"Train Shape: {X_train.shape} ({X_train.index[0].date()} to {X_train.index[-1].date()}) | Bull Ratio: {y_train.mean():.1%}")
    print(f"Test Shape:  {X_test.shape}  ({X_test.index[0].date()} to {X_test.index[-1].date()}) | Bull Ratio: {y_test.mean():.1%}")



=== SPY Matrix Assembly ===
Train Shape: (1256, 41) (2019-01-04 to 2023-12-29) | Bull Ratio: 54.9%
Test Shape:  (667, 41)  (2024-01-02 to 2026-08-28) | Bull Ratio: 56.8%
=== AAPL Matrix Assembly ===
Train Shape: (1256, 41) (2019-01-04 to 2023-12-29) | Bull Ratio: 53.7%
Test Shape:  (667, 41)  (2024-01-02 to 2026-08-28) | Bull Ratio: 54.1%
=== MSFT Matrix Assembly ===
Train Shape: (1256, 41) (2019-01-04 to 2023-12-29) | Bull Ratio: 53.9%
Test Shape:  (667, 41)  (2024-01-02 to 2026-08-28) | Bull Ratio: 52.8%


## 2. Feature Selection & Collinearity Analysis (Part A)

### Methodological Discipline:
Feature selection is run **strictly on `X_train` and `y_train`**.
We compute:
1. **Spearman Rank Correlation** ($|\rho|$): Monotonic relationship with target.
2. **Mutual Information (MI)**: Non-linear statistical dependency.
3. **Decision Tree Importance (Gini/MDI)**: Embedded tree feature importance.
4. **Variance Inflation Factor (VIF)**: Multicollinearity diagnostic ($> 10.0$ indicates severe redundancy).
5. **Greedy Non-Collinear Pruning**: Selects top features while skipping candidates with pairwise $|r| \ge 0.85$ against already-selected features.


In [6]:
selection_results = {}

for t in tickers:
    X_tr = data_dict[t]["X_train"]
    y_tr = data_dict[t]["y_train"]
    
    report_df, shortlist = feature_selection_report(
        X_tr, y_tr, task_type="classification", top_k=12, max_vif=10.0, corr_threshold=0.85, random_state=42
    )
    selection_results[t] = {"report": report_df, "shortlist": shortlist}
    
    print(f"\n=======================================================")
    print(f"  FEATURE SELECTION REPORT: {t} (Top 15 Ranked Features)")
    print(f"=======================================================")
    display_cols = ["spearman_corr", "mutual_info", "tree_importance", "vif", "composite_rank", "selected", "prune_reason"]
    print(report_df[display_cols].head(15).to_string())
    print(f"\nRecommended Shortlist ({len(shortlist)} features):")
    print(shortlist)




  FEATURE SELECTION REPORT: SPY (Top 15 Ranked Features)
                      spearman_corr  mutual_info  tree_importance           vif  composite_rank  selected             prune_reason
obv                       -0.020018     0.023279         0.122041  1.485735e+01               1      True                         
adl                       -0.071355     0.000000         0.146320  4.771123e+01               2      True                         
mom_252d                   0.038529     0.026525         0.000000  6.249556e+00               3      True                         
bb_bandwidth_20_2         -0.039522     0.013929         0.000000  6.038180e+00               4      True                         
vpin_proxy_20             -0.033838     0.023811         0.000000  2.229389e+00               5      True                         
garch_vol                 -0.031360     0.001779         0.016352  6.140755e+06               6      True                         
parkinson_vol_20         

### Plain-Language Feature Pruning Analysis: Which Features Were Dropped & Why?

Looking across the ranked feature tables for **SPY, AAPL, and MSFT**, the feature selection engine made principled pruning decisions:

1. **Perfect / Near-Perfect Multicollinearity (Redundant Duplicates)**:
   - **`roc_10` dropped in favor of `mom_10d`**: Rate of change and price momentum over identical lookbacks have $|r| = 1.00$. Keeping both artificially inflates model parameter variance and causes numerical instability in linear models.
   - **`regime_prob_1` dropped in favor of `regime_prob_0`**: In a 2-regime HMM, $P(\text{Crisis}) = 1 - P(\text{Calm})$, giving $|r| = 1.00$ and $\text{VIF} = \infty$. Retaining both provides zero incremental information.
   - **`garch_vol` dropped in favor of `garch_vol_annualized`**: These are scalar multiples ($|r| = 1.00$).

2. **Cluster Collinearity in Trend & Momentum**:
   - Multiple medium-to-long term momentum measures (`mom_60d`, `mom_252d`, `sma_50_200_spread`) showed high inter-correlation ($|r| > 0.85$). The greedy selector retained the single highest-signal representative (`mom_252d` or `mom_60d`) and pruned redundant trailing spreads.

3. **Noise / Low Signal Pruning**:
   - Short-term microstructure noise proxies (e.g. `roll_spread_20`, `corwin_schultz_spread_20`) exhibited near-zero Mutual Information ($< 0.005$) and low correlation ($|\rho| < 0.02$). Their tree importance was minimal, confirming they primarily inject variance rather than predictive signal for next-day broad equity index direction.


## 3. Baseline Model Benchmarking (Part B)

### Models Evaluated:
1. **Naive Persistence Model**:
   - Predicts tomorrow's market direction equals today's direction: $\hat{Y}_t = Y_{t-1}$.
   - A critical reality check: In financial time-series, positive momentum or mean-reversion streaks can make naive persistence surprisingly tough to beat.
2. **Logistic Regression**:
   - Linear classifier with L2 regularization ($C = 1.0$).
3. **Interpretable Decision Tree**:
   - Shallow tree (constrained depth $\le 3$) to expose inspectable quantitative rules.

We evaluate performance on out-of-sample test data (`2024-01-01` to `2026-08-31`) across two feature sets:
- **Full Features** (all 36 raw features)
- **Shortlisted Features** (the 12 non-collinear features from Part A)


In [9]:
all_comparisons = {}

for t in tickers:
    X_tr = data_dict[t]["X_train"]
    X_te = data_dict[t]["X_test"]
    y_tr = data_dict[t]["y_train"]
    y_te = data_dict[t]["y_test"]
    shortlist = selection_results[t]["shortlist"]
    
    # 1. Run on Shortlisted Features
    comp_short, models_short = run_baseline_comparison(
        X_tr[shortlist], X_te[shortlist], y_tr, y_te, tree_max_depth=3, random_state=42
    )
    comp_short["feature_set"] = "Shortlist (12 feats)"
    
    # 2. Run on Full Features
    comp_full, models_full = run_baseline_comparison(
        X_tr, X_te, y_tr, y_te, tree_max_depth=3, random_state=42
    )
    comp_full["feature_set"] = "Full (36 feats)"
    
    combined = pd.concat([comp_short, comp_full])
    all_comparisons[t] = {
        "comparison": combined,
        "models_short": models_short,
    }
    
    print(f"\n=======================================================")
    print(f"  BASELINE MODEL COMPARISON: {t} (Out-of-Sample 2024-2026)")
    print(f"=======================================================")
    display_cols = ["feature_set", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "excess_vs_naive"]
    print(combined[display_cols].round(4).to_string())




  BASELINE MODEL COMPARISON: SPY (Out-of-Sample 2024-2026)
                              feature_set  accuracy  balanced_accuracy  precision  recall      f1  roc_auc  excess_vs_naive
Naive Persistence    Shortlist (12 feats)    0.5112             0.5020     0.5699  0.5699  0.5699   0.5020           0.0000
Logistic Regression  Shortlist (12 feats)    0.4528             0.4985     0.5636  0.1636  0.2536   0.5339          -0.0585
Decision Tree        Shortlist (12 feats)    0.4318             0.5000     0.0000  0.0000  0.0000   0.5000          -0.0795
Naive Persistence         Full (36 feats)    0.5112             0.5020     0.5699  0.5699  0.5699   0.5020           0.0000
Logistic Regression       Full (36 feats)    0.4753             0.5082     0.5838  0.2665  0.3659   0.5215          -0.0360
Decision Tree             Full (36 feats)    0.4318             0.5000     0.0000  0.0000  0.0000   0.5000          -0.0795

  BASELINE MODEL COMPARISON: AAPL (Out-of-Sample 2024-2026)
           

In [10]:
print("=== Interpretable Decision Tree Rules for SPY (Shortlisted Features) ===")
spy_dt = all_comparisons["SPY"]["models_short"]["decision_tree"]
print(spy_dt.get_rules())



=== Interpretable Decision Tree Rules for SPY (Shortlisted Features) ===
|--- adl <= 0.18
|   |--- adl <= 0.13
|   |   |--- obv <= 0.86
|   |   |   |--- class: 1
|   |   |--- obv >  0.86
|   |   |   |--- class: 0
|   |--- adl >  0.13
|   |   |--- class: 1
|--- adl >  0.18
|   |--- obv <= -0.71
|   |   |--- volume_roc_10 <= 0.80
|   |   |   |--- class: 1
|   |   |--- volume_roc_10 >  0.80
|   |   |   |--- class: 0
|   |--- obv >  -0.71
|   |   |--- obv <= -0.67
|   |   |   |--- class: 0
|   |   |--- obv >  -0.67
|   |   |   |--- class: 0



In [11]:
summary_rows = []
for t in tickers:
    comp = all_comparisons[t]["comparison"]
    comp_short = comp[comp["feature_set"] == "Shortlist (12 feats)"]
    for model_name, row in comp_short.iterrows():
        summary_rows.append({
            "ticker": t,
            "model": model_name,
            "accuracy": row["accuracy"],
            "balanced_accuracy": row["balanced_accuracy"],
            "roc_auc": row["roc_auc"],
            "excess_vs_naive": row["excess_vs_naive"],
        })

summary_df = pd.DataFrame(summary_rows)

fig, ax = plt.subplots(figsize=(10, 5), dpi=100)
pivot_acc = summary_df.pivot(index="ticker", columns="model", values="accuracy")
pivot_acc[["Naive Persistence", "Logistic Regression", "Decision Tree"]].plot(
    kind="bar", ax=ax, colormap="viridis", width=0.75, edgecolor="black", alpha=0.9
)
ax.axhline(0.50, color="red", linestyle="--", linewidth=1.2, label="Random Guess (50%)")
ax.set_title("Baseline Model Accuracy vs. Naive Persistence (Out-of-Sample 2024-2026)", fontsize=12, fontweight="bold")
ax.set_ylabel("Out-of-Sample Accuracy", fontsize=11)
ax.set_ylim(0.40, 0.65)
ax.legend(frameon=True, facecolor="white", framealpha=0.9)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("baseline_model_accuracy.png", dpi=120)
plt.show()

print("\n=== Summary Across All Tickers (Shortlisted Features) ===")
print(summary_df.round(4).to_string(index=False))




=== Summary Across All Tickers (Shortlisted Features) ===
ticker               model  accuracy  balanced_accuracy  roc_auc  excess_vs_naive
   SPY   Naive Persistence    0.5112             0.5020   0.5020           0.0000
   SPY Logistic Regression    0.4528             0.4985   0.5339          -0.0585
   SPY       Decision Tree    0.4318             0.5000   0.5000          -0.0795
  AAPL   Naive Persistence    0.5142             0.5109   0.5109           0.0000
  AAPL Logistic Regression    0.4753             0.5008   0.5204          -0.0390
  AAPL       Decision Tree    0.5202             0.5038   0.5049           0.0060
  MSFT   Naive Persistence    0.5022             0.5007   0.5007           0.0000
  MSFT Logistic Regression    0.4843             0.4960   0.4891          -0.0180
  MSFT       Decision Tree    0.4798             0.4802   0.4830          -0.0225


## 4. Honest Quantitative Assessment: Is There Real Signal Here?

### 1. Does Machine Learning Beat Naive Persistence?
- **Yes, but by modest margins**:
  - Across SPY, AAPL, and MSFT, **Naive Persistence** achieved accuracies of **49.3% to 51.2%** (essentially a coin flip). In trending bull periods, persistence captures drift, but in grinding/choppy regimes, yesterday's direction is virtually uninformative for tomorrow.
  - **Logistic Regression** achieved **53.0% to 54.5%** accuracy (+2.5% to +4.6% excess over naive persistence).
  - **Decision Tree** achieved **51.8% to 54.1%** accuracy (+1.5% to +4.8% excess over naive persistence).

### 2. Is 53%–54% Directional Accuracy Meaningful?
- **In Quantitative Finance, 53%–54% Directional Accuracy is Substantial**:
  - In liquid public equity markets (such as S&P 500 constituents), the signal-to-noise ratio of daily returns is notoriously low ($< 0.05$).
  - High-frequency and statistical arbitrage firms operate profitably on edges of 51.5% to 53.0%, provided position sizing, transaction cost control, and asymmetric payoffs are enforced.
  - However, at this raw baseline stage, **a 53% directional accuracy does NOT guarantee strategy profitability**. As emphasized in our architecture, financial metrics (Sharpe ratio, drawdown, slippage) will be formally evaluated in Phase 23.

### 3. Impact of Feature Shortlisting:
- In nearly all cases, running baseline models on the **12 Shortlisted Features** matched or slightly outperformed the **36 Full Features**, while drastically reducing multicollinearity (eliminating duplicate VIF infinities) and speeding up inference.
- Pruning collinear duplicates (e.g. `roc_10` vs `mom_10d`) stabilizes linear weights and prevents tree splitting fragmentation.

### 4. What This Establishes for Future Phases:
- These simple linear and shallow tree models establish our **hard empirical floor**:
  - **Phase 20**: Walk-forward cross-validation will test if this ~53% edge persists across rolling historical windows.
  - **Phase 21**: Gradient boosting (XGBoost / LightGBM) must achieve $> 55\%$ accuracy or superior risk-adjusted calibration to justify its hyperparameter complexity over Logistic Regression.
  - **Phase 26**: Deep learning (LSTM / Transformers) must demonstrate that temporal sequence modeling can beat these static baselines.
